# Demo: reading the sparsity rate of TEAL and WINA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/sparsity_rates_teal_wina.ipynb)

TEAL and WINA make an LLM faster by setting part of its activations to zero. They are **threshold
methods**: in each layer, an activation is cut when its score is below a threshold. So we do not choose
how many activations are cut: it depends on the token and on the layer, and is only known after the
forward pass. This notebook shows, on two example prompts, how to read it with a PyTorch hook: the
sparsity of each layer (local) and of the whole model (global).

This is a basic demonstrator. The real measurements (whole dataset, unquantized model) will be done in
the experiments.

**How to run it**

1. Open the notebook in Colab: click the badge above, or *File → Open notebook →
   GitHub*, paste the repo URL, pick your branch and select this notebook.
2. Select a GPU: *Runtime → Change runtime type → GPU* (the free T4 is enough).
3. If you are testing a branch, set `BRANCH` in the setup cell below.
4. *Runtime → Run all*. The first run takes ~5 min to download Qwen3-8B (16 GB).

## 0. Setup

This cell does the following:
- clone the repo on Colab and check out the selected `BRANCH`.
- install the dependencies on Colab.

Rerun this cell every time you open a Colab session and after every push on GitHub.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # put your branch here to test code that is not merged yet

if IN_COLAB:
    # Clone the repo (or update it), then install the procom package and its dependencies
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # makes procom importable without restarting

In [ ]:
import torch

import procom  # noqa: F401  (checks that the repo's package is installed)

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## 1. Loading Qwen3-8B

Qwen3-8B in bf16 takes ~16 GB: it does not fit on a T4 (15 GB), so in that case we load it in 8-bit.
Quantization slightly changes the activations, which is fine for a demo (not for the experiments).
The `down_proj` layers are kept in bf16 even then: their weights are used by WINA (section 2).

In each layer, the FFN computes `down_proj(a)` with `a = SiLU(gate_proj(x)) * up_proj(x)`, a vector of
12,288 values, one per **neuron**. This vector `a` is what TEAL and WINA make sparse: setting `a_j` to
zero means column `j` of `down_proj`'s weight matrix is not needed for this token.

In [ ]:
import logging

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
load_in_8bit = gpu_memory_gb < 20
# down_proj stays in bf16 (WINA reads its weights); lm_head is never quantized by default
quantization = (
    BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=["down_proj", "lm_head"])
    if load_in_8bit
    else None
)
# In 8-bit, bitsandbytes logs a harmless "inputs will be cast to float16" warning at every
# forward pass: we hide it (and only bitsandbytes warnings)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto", quantization_config=quantization
)
model.eval()
torch.set_grad_enabled(False)
device = model.device
layers = model.model.layers  # the transformer blocks; each has its FFN in `.mlp`
N_LAYERS = len(layers)
assert layers[0].mlp.down_proj.weight.is_floating_point(), "down_proj must not be quantized"

print(f"GPU {gpu_memory_gb:.0f} GB | 8-bit: {load_in_8bit} | {N_LAYERS} layers")

## 2. Which neurons are cut: TEAL and WINA

Both methods give each neuron a score and cut the neurons with the lowest scores.

- **TEAL** ([Liu et al., 2025](https://arxiv.org/abs/2408.14690)) looks only at the activation: the
  score of neuron `j` is `|a_j|`. A small activation is assumed to matter little.
- **WINA** ([Chen et al., 2025](https://arxiv.org/abs/2505.19427)) also looks at the weights the
  activation is multiplied by: the score is `|a_j| × ‖w_j‖`, where `w_j` is column `j` of
  `down_proj`'s weights. It measures how much the neuron really changes the layer's output.

Example with two neurons:

| | activation `a_j` | weight norm `‖w_j‖` | TEAL score | WINA score |
|---|---|---|---|---|
| neuron 1 | 0.8 | 0.1 | 0.8 | 0.08 |
| neuron 2 | 0.3 | 2.0 | 0.3 | 0.6 |

TEAL cuts neuron 2 first (smaller activation). WINA cuts neuron 1 first: its activation is large, but
it is multiplied by small weights, so it barely changes the output.

(WINA also has an optional rotation of the weights, which we do not use: it changes the basis, and the
neurons would no longer be those of the original model.)

In [ ]:
METHODS = ("teal", "wina")

# ‖w_j‖ for every layer: the norm of each column of down_proj's weights (used by WINA)
COLUMN_NORMS = [layer.mlp.down_proj.weight.float().norm(dim=0) for layer in layers]


def neuron_scores(acts, method, layer):
    """Score of every neuron (same shape as `acts`): the lowest ones are cut."""
    scores = acts.abs().float()
    if method == "wina":
        scores = scores * COLUMN_NORMS[layer].to(scores.device)
    return scores

## 3. Choosing the thresholds

The papers do not give a threshold value: the right one depends on the model and on the layer. They
give a **target sparsity** instead, and TEAL derives each layer's threshold from it: it runs the model
on some text and takes, in each layer, the score below which that fraction of the activations falls
(a quantile). TEAL reports 40-50 % sparsity over the whole model with little loss of quality, so we
target **50 %**.

The thresholds are computed once on a short generic text. On other prompts, the fraction cut will no
longer be exactly 50 %: this is the realized sparsity we want to read.

In [ ]:
TARGET_SPARSITY = 0.5

CALIBRATION_TEXT = (
    "The river runs through the old town, past the market square and the stone bridge built in "
    "the twelfth century. Every Saturday, farmers from the surrounding villages sell vegetables, "
    "cheese and bread there. In 1998 the town council decided to close the centre to cars, and "
    "the number of visitors has grown every year since. Scientists who study the river have "
    "measured its temperature for decades: summers are warmer than they used to be, and some "
    "fish species have moved further north. The local school uses these measurements to teach "
    "children how to read a graph, compute an average and question the conclusions of a study."
)


def calibrate_thresholds(method, text=CALIBRATION_TEXT, sparsity=TARGET_SPARSITY):
    """One threshold per layer: the score below which a fraction `sparsity` of `text` falls."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    thresholds = [None] * N_LAYERS

    def make_hook(layer):
        def hook(module, args):  # reads the input of down_proj, without changing it
            scores = neuron_scores(args[0], method, layer)
            thresholds[layer] = scores.flatten().quantile(sparsity).item()

        return hook

    handles = [
        block.mlp.down_proj.register_forward_pre_hook(make_hook(i))
        for i, block in enumerate(layers)
    ]
    try:
        model(**inputs)
    finally:
        for handle in handles:
            handle.remove()  # always remove the hooks, otherwise they stay active afterwards
    return thresholds


thresholds = {method: calibrate_thresholds(method) for method in METHODS}

## 4. The hook that cuts and counts

A *forward pre-hook* on `down_proj` receives its input `a` before the layer runs and can replace it.
In every layer, our hook sets to zero the neurons whose score is below the layer's threshold, and
counts, for each token, the fraction of the layer's neurons it cut.

In [ ]:
def run_sparse(messages, method):
    """Runs Qwen3-8B on a conversation with `method`'s thresholds.

    Returns the tokens and `cut`, of shape (n_layers, n_tokens): cut[l, t] is the fraction of
    layer l's neurons set to zero for token t.
    """
    text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    cut = [None] * N_LAYERS

    def make_hook(layer):
        def hook(module, args):
            acts = args[0]
            keep = neuron_scores(acts, method, layer) >= thresholds[method][layer]
            cut[layer] = 1 - keep[0].float().mean(dim=-1).cpu()  # one value per token
            return (acts * keep,)  # replaces the input of down_proj

        return hook

    handles = [
        block.mlp.down_proj.register_forward_pre_hook(make_hook(i))
        for i, block in enumerate(layers)
    ]
    try:
        model(**inputs)
    finally:
        for handle in handles:
            handle.remove()
    tokens = [tokenizer.decode(t) for t in inputs.input_ids[0]]
    return tokens, torch.stack(cut)

## 5. Reading the sparsity on two examples

- **Local sparsity** of a layer: fraction of its neurons cut, averaged over the prompt's tokens.
- **Global sparsity**: fraction of all the FFN neurons of the model cut. Every layer has the same number
  of neurons, so it is the mean of the local sparsities.

In [ ]:
examples = {
    "fact": [{"role": "user", "content": "The Eiffel Tower is located in the city of"}],
    "calculation": [{"role": "user", "content": "What is 23 + 48? Answer with the number only."}],
}

cuts = {}
for name, messages in examples.items():
    for method in METHODS:
        tokens, cut = run_sparse(messages, method)
        cuts[name, method] = cut
        print(f"{name:12} {method.upper():5} global sparsity: {cut.mean():.1%}")

Local sparsity of each layer, for each example and method. The target was 50 % on the calibration
text: the realized values move around it, differently in each layer.

In [ ]:
import pandas as pd

local = pd.DataFrame(
    {f"{name} / {method.upper()}": cut.mean(dim=1) for (name, method), cut in cuts.items()}
)
local.index.name = "layer"
print(local.map("{:.0%}".format).to_string())

The fraction cut also changes from one token to the next. Global sparsity of each token of the first
example, with TEAL:

In [ ]:
tokens, cut = run_sparse(examples["fact"], "teal")
for token, value in zip(tokens, cut.mean(dim=0), strict=True):
    print(f"{value:6.1%}  {token!r}")